# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# the .metadata object provides schema.org and Croissant fields directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"Authors (IDs): {[a['@id'] if isinstance(a, dict) and '@id' in a else a for a in dataset.metadata.author]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets available in the dataset
metadata_json = dataset.metadata.to_json()  # machine-readable full Croissant metadata

# Gather RecordSets with their @ids
record_sets = metadata_json.get('recordSet', [])
if not record_sets:
    raise RuntimeError("No record sets found in the dataset metadata. The dataset might be single-table and the main file represents the only recordset.")

print('Found record sets:')
for rs in record_sets:
    if isinstance(rs, dict):
        rs_id = rs.get('@id', str(rs))
        print(f"  RecordSet @id: {rs_id}")
        fields = rs.get('field', [])
        print("    Fields (@id):")
        for f in fields:
            if isinstance(f, dict):
                print(f"      - {f.get('@id', f)}")
            else:
                print(f"      - {f}")
    else:
        print(f"  RecordSet: {rs}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# --- Identify record set(s) to extract ---
# If only one record set is present and not entered via @id reference, use the first available.
if record_sets:
    ids = []
    for rs in record_sets:
        if isinstance(rs, dict) and '@id' in rs:
            ids.append(rs['@id'])
        elif isinstance(rs, str):
            ids.append(rs)
    record_sets_ids = ids
else:
    raise RuntimeError("No record sets found in metadata.")

# Load data from each record set
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for RecordSet @id={record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set {record_set_id}.")

# Display columns for the first record set
main_recordset_id = record_sets_ids[0]
if main_recordset_id in dataframes:
    print(f"Columns in DataFrame for record set {main_recordset_id}:")
    print(dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# --- Example: Analyze numeric and group columns in the main record set ---
df = dataframes[main_recordset_id]

# Suggest or pick a numeric field (pick the first one that looks numeric)
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    # Try to coerce columns to numeric and use one with >50% able to parse
    for col in df.columns:
        coerced = pd.to_numeric(df[col], errors='coerce')
        if coerced.notnull().mean() > 0.5:
            numeric_field = col
            df[col] = coerced
            break

if numeric_field:
    print(f"Using numeric field '@id' or column name: {numeric_field}")
    threshold = df[numeric_field].quantile(0.75)
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (75th percentile):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Pick a categorical/group-by field (non-numeric)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            unique_vals = df[col].nunique()
            if 1 < unique_vals < 20:
                group_field = col
                break
    if group_field:
        print(f"Grouping by field '@id' or column name: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"Grouped mean {numeric_field} by {group_field}: ")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells, examining its metadata, record set(s), and columns (referenced by their Croissant `@id`).

- We identified available record sets and fields using their `@id` to keep references precise and reproducible.
- We filtered and normalized numerical fields, grouped by categorical columns, and visualized the structure of the main record set.

Further analyses are possible by extending this notebook to feature engineering, advanced statistical testing, and machine learning applications specific to the biological domain of the dataset.